# AI Model Experiment & Evaluation

## 1. Problem Statement

### Objective

Eksperimen ini bertujuan membandingkan dua pendekatan AI untuk mengklasifikasikan sentimen ulasan pelanggan e-commerce, yaitu model Machine Learning klasik menggunakan Scikit-learn dan Large Language Model (LLM) yang diakses melalui Gemini API.

### Target

Target prediksi adalah kolom `sentiment`, dengan dua kelas: `positif` dan `negatif`. Kolom `review_text` digunakan sebagai input utama untuk memprediksi sentimen.

### Batasan dan Asumsi

- Eksperimen menggunakan dataset yang diberikan dalam assignment.
- Kedua pendekatan dievaluasi pada test set yang sama.
- Model klasik dilatih menggunakan data training, sedangkan Gemini menggunakan pendekatan prompting tanpa training khusus pada dataset.
- Rekomendasi akhir didasarkan pada hasil evaluasi, kemudahan implementasi, waktu inferensi, dan potensi biaya.


## 2. Import Library & Load Dataset

Pada tahap ini, kita mengimpor library yang diperlukan dan memuat dataset ulasan pelanggan menggunakan Pandas. Setelah dataset dimuat, kita akan memeriksa struktur dan kualitas data sebelum melakukan eksperimen model.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import os
import time

In [2]:
df = pd.read_csv("../data/customer_reviews_sentiment.csv")
df.head()

,review_id,product_name,review_text,sentiment
0,1,Kemeja Flanel,Pelayanan lambat dan tidak responsif saat diko...,negatif
1,2,Case HP,"Puas banget belanja disini, proses cepat dan b...",positif
2,3,Rice Cooker,"Terima kasih seller, barangnya awet dan sesuai...",positif
3,4,Case HP,Warna produk berbeda jauh dari foto di listing.,negatif
4,5,Headset Bluetooth,"Terima kasih seller, barangnya awet dan sesuai...",positif


In [3]:
print("Jumlah baris dan kolom:", df.shape)
print("Jumlah baris:", df.shape[0])
print("Jumlah kolom:", df.shape[1])

Jumlah baris dan kolom: (200, 4)
Jumlah baris: 200
Jumlah kolom: 4


In [4]:
print("Nama Kolom:")
print(df.columns.tolist())

print("\nTipe data:")
print(df.dtypes)

Nama Kolom:
['review_id', 'product_name', 'review_text', 'sentiment']

Tipe data:
review_id       int64
product_name      str
review_text       str
sentiment         str
dtype: object


In [5]:
print("Total missing values:")
print(df.isnull().sum())
print("\nTotal duplicated rows:")
print(df.duplicated().sum())

Total missing values:
review_id       0
product_name    0
review_text     0
sentiment       0
dtype: int64

Total duplicated rows:
0


In [6]:
print("Distribusi sentimen:")
print(df["sentiment"].value_counts())

print("\nProporsi sentiment:")
print(df["sentiment"].value_counts(normalize=True))

Distribusi sentimen:
sentiment
positif    110
negatif     90
Name: count, dtype: int64

Proporsi sentiment:
sentiment
positif    0.55
negatif    0.45
Name: proportion, dtype: float64


In [7]:
empty_reviews = df["review_text"].str.strip().eq("").sum()
print("Jumlah ulasan kosong", empty_reviews)
print("\nLabel sentimen unik:", df["sentiment"].unique())

Jumlah ulasan kosong 0

Label sentimen unik: <StringArray>
['negatif', 'positif']
Length: 2, dtype: str


## 3. Menyiapkan Train/Test Split

Dataset dibagi menjadi training set dan test set dengan proporsi 80:20. Training set digunakan untuk melatih model Machine Learning klasik, sedangkan test set digunakan untuk mengevaluasi performa kedua pendekatan AI.

Parameter stratify digunakan untuk menjaga proporsi kelas sentimen agar relatif konsisten pada kedua subset.

In [8]:
X = df["review_text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 160
Test size: 40


## 4. Pendekatan 1 — Model Klasik (Scikit-learn)

Pada pendekatan pertama, klasifikasi sentimen dilakukan menggunakan Logistic Regression. Karena model Machine Learning klasik tidak dapat memproses teks mentah secara langsung, teks ulasan akan diubah menjadi representasi numerik menggunakan TF-IDF.

Eksperimen ini menggunakan training set untuk membangun model dan test set untuk mengevaluasi prediksi.

### 4.1 Preprocessing & Feature Extraction

TF-IDF (*Term Frequency–Inverse Document Frequency*) digunakan untuk mengubah teks ulasan menjadi fitur numerik berdasarkan frekuensi kata dalam ulasan dan tingkat keumuman kata tersebut dalam kumpulan dokumen.

In [9]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Training feature matrix:", X_train_vec.shape)
print("Testing feature matrix:", X_test_vec.shape)
print("Jumlah vocabulary:")
print(len(vectorizer.vocabulary_))

Training feature matrix: (160, 145)
Testing feature matrix: (40, 145)
Jumlah vocabulary:
145


### 4.2 Training Model

Logistic Regression digunakan sebagai algoritma klasifikasi untuk mempelajari hubungan antara fitur TF-IDF dan label sentimen. Model dilatih menggunakan data training yang telah diubah menjadi representasi numerik.

In [10]:
clf = LogisticRegression()

clf.fit(X_train_vec,y_train)

print("Model training selesai.")

Model training selesai.


In [11]:
y_pred_classic = clf.predict(X_test_vec)

print("Jumlah prediksi:", len(y_pred_classic))
print("\nLima prediksi pertama:")
print(y_pred_classic[:5])

Jumlah prediksi: 40

Lima prediksi pertama:
['negatif' 'positif' 'positif' 'positif' 'negatif']


### 4.3 Evaluasi Model Klasik

Performa model Logistic Regression dievaluasi menggunakan Accuracy, Precision, Recall, dan F1-Score. Selain itu, confusion matrix digunakan untuk melihat jumlah prediksi yang benar dan salah pada setiap kelas sentimen.

Metrik evaluasi dihitung menggunakan prediksi pada test set dan label sentimen sebenarnya. Karena kelas yang digunakan adalah `positif` dan `negatif`, parameter `pos_label="positif"` digunakan untuk menentukan kelas positif dalam perhitungan Precision, Recall, dan F1-Score.

In [12]:
accuracy_classic = accuracy_score(y_test, y_pred_classic)

precision_classic = precision_score(
    y_test, y_pred_classic, pos_label="positif"
)

recall_classic = recall_score(
    y_test, y_pred_classic, pos_label="positif"
)

f1_classic = f1_score(
    y_test, y_pred_classic, pos_label="positif"
)

print("=== Evaluasi Model Klasik ===")
print(f"Accuracy : {accuracy_classic:.4f}")
print(f"Precision: {precision_classic:.4f}")
print(f"Recall   : {recall_classic:.4f}")
print(f"F1-Score : {f1_classic:.4f}")

=== Evaluasi Model Klasik ===
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1-Score : 1.0000


In [13]:
cm_classic = confusion_matrix(
    y_test,
    y_pred_classic,
    labels=["negatif", "positif"]
)

cm_classic_df = pd.DataFrame(
    cm_classic,
    index=["Aktual Negatif", "Aktual Positif"],
    columns=["Prediksi Negatif", "Prediksi Positif"]
)

cm_classic_df

,Prediksi Negatif,Prediksi Positif
Aktual Negatif,18,0
Aktual Positif,0,22


In [14]:
print(classification_report(y_test,y_pred_classic))

              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00        18
     positif       1.00      1.00      1.00        22

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



## 5. Pendekatan 2 — Gemini API

Pada pendekatan kedua, klasifikasi sentimen dilakukan menggunakan Large Language Model (LLM) melalui Gemini API. Berbeda dengan model klasik yang dilatih menggunakan training set, Gemini melakukan klasifikasi berdasarkan instruksi (*prompt*) yang diberikan.

Setiap ulasan dari test set akan dikirimkan kepada Gemini untuk memperoleh prediksi sentimen. Prediksi tersebut kemudian dibandingkan dengan label aktual pada test set yang sama seperti yang digunakan untuk mengevaluasi model klasik.

**Catatan konfigurasi Gemini API**

Pada tahap eksperimen awal, Gemini API diinisialisasi menggunakan API key melalui environment variable `GEMINI_API_KEY`. API key tidak disimpan di dalam notebook.

Untuk menjaga agar notebook final dapat dijalankan ulang tanpa menghabiskan kuota API, hasil prediksi Gemini yang telah diperoleh pada eksperimen awal dimuat dari file `../data/gemini_partial_results.csv`. Dengan demikian, cell berikutnya tidak mengirim request baru ke Gemini API.


### 5.1 Pengujian Koneksi Gemini API

Koneksi dan respons Gemini API telah diuji pada tahap eksperimen awal. Pengujian tersebut memastikan bahwa API key, SDK, dan model yang digunakan dapat menghasilkan respons.

Pengujian koneksi tidak diulang pada notebook final agar proses reproduksi hasil tidak menggunakan kuota API tambahan.


### 5.2 Fungsi Klasifikasi Sentimen

Fungsi klasifikasi berbasis prompt digunakan pada eksperimen awal untuk meminta Gemini menentukan sentimen setiap ulasan. Hasil prediksi yang berhasil diperoleh kemudian disimpan agar dapat dievaluasi kembali tanpa mengirim request API berulang kali.


In [ ]:
def predict_sentiment_gemini(review):
    prompt = f""" 

        Kamu adalah sistem klasifikasi sentimen ulasan pelanggan.

        Tugas:
        Tentukan sentimen dari ulasan berikut.

        Aturan:
        - Jawab hanya dengan satu kata: positif atau negatif.
        - Jika ulasan menunjukkan kepuasan, pengalaman baik, atau penilaian positif, jawab positif.
        - Jika ulasan menunjukkan ketidakpuasan, pengalaman buruk, atau penilaian negatif, jawab negatif.
        - Jangan memberikan penjelasan tambahan.

        Ulasan:
        {review}

        Sentimen:
        """

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )

    prediction = response.text.strip().lower()

    if prediction not in ["positif", "negatif"]:
        raise ValueError(
            f"Respons Gemini tidak sesuai format: {prediction}"
        )

    return prediction

Fungsi `predict_sentiment_gemini()` telah diuji pada tahap eksperimen awal menggunakan ulasan berlabel negatif. Fungsi menghasilkan prediksi `negatif` sesuai dengan isi ulasan. Pengujian fungsi tidak diulang pada notebook final untuk menghindari penggunaan kuota API tambahan.


### 5.3 Prediksi pada Test Set

Fungsi klasifikasi yang telah diuji kemudian diterapkan pada seluruh ulasan dalam test set. Setiap ulasan dikirimkan ke Gemini API untuk memperoleh prediksi sentimen. Seluruh prediksi disimpan untuk dibandingkan dengan label aktual menggunakan metrik evaluasi yang sama dengan model klasik.

Waktu inferensi juga diukur untuk mengetahui durasi yang diperlukan Gemini dalam memproses test set.

In [ ]:
import pandas as pd

# Memuat hasil prediksi Gemini yang telah disimpan pada eksperimen awal
gemini_partial_df = pd.read_csv("../data/gemini_partial_results.csv")

# Memastikan file berisi hasil prediksi yang diharapkan
if len(gemini_partial_df) != 21:
    raise ValueError(
        f"Jumlah prediksi Gemini pada CSV adalah {len(gemini_partial_df)}, "
        "seharusnya 21. Periksa file gemini_partial_results.csv."
    )

gemini_results = dict(
    zip(
        gemini_partial_df["index"].astype(int),
        gemini_partial_df["predicted_sentiment"].astype(str)
    )
)

# Waktu ini berasal dari catatan eksperimen awal dan mencakup jeda antarrequest
# serta komunikasi API, bukan waktu inferensi murni model.
gemini_total_processing_time = 538.79

print("Prediksi Gemini berhasil dimuat:", len(gemini_results), "/", len(X_test))
print(f"Total waktu pemrosesan eksperimen sebelumnya: {gemini_total_processing_time:.2f} detik")


Prediksi Gemini berhasil dimuat: 21 / 40
Total waktu pemrosesan eksperimen sebelumnya: 538.79 detik


In [20]:

# Mengubah dictionary hasil Gemini menjadi Pandas Series
y_pred_gemini = pd.Series(gemini_results, name="predicted_sentiment")

# Mengambil label aktual berdasarkan index prediksi Gemini
y_test_gemini = y_test.loc[y_pred_gemini.index]

# Menghitung metrik evaluasi
accuracy_gemini = accuracy_score(y_test_gemini, y_pred_gemini)

precision_gemini = precision_score(
    y_test_gemini,
    y_pred_gemini,
    pos_label="positif",
    zero_division=0
)

recall_gemini = recall_score(
    y_test_gemini,
    y_pred_gemini,
    pos_label="positif",
    zero_division=0
)

f1_gemini = f1_score(
    y_test_gemini,
    y_pred_gemini,
    pos_label="positif",
    zero_division=0
)

print("=== Evaluasi Gemini ===")
print("Jumlah data evaluasi:", len(y_pred_gemini))
print(f"Accuracy : {accuracy_gemini:.4f}")
print(f"Precision: {precision_gemini:.4f}")
print(f"Recall   : {recall_gemini:.4f}")
print(f"F1-Score : {f1_gemini:.4f}")


=== Evaluasi Gemini ===
Jumlah data evaluasi: 21
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1-Score : 1.0000


Confusion matrix digunakan untuk melihat distribusi hasil klasifikasi Gemini berdasarkan label aktual dan label prediksi. Evaluasi dilakukan terhadap 21 data yang berhasil diproses oleh Gemini.

In [21]:

# Membuat confusion matrix Gemini
cm_gemini = confusion_matrix(
    y_test_gemini,
    y_pred_gemini,
    labels=["negatif", "positif"]
)

# Mengubah hasil menjadi DataFrame agar mudah dibaca
cm_gemini_df = pd.DataFrame(
    cm_gemini,
    index=["Aktual Negatif", "Aktual Positif"],
    columns=["Prediksi Negatif", "Prediksi Positif"]
)

cm_gemini_df


,Prediksi Negatif,Prediksi Positif
Aktual Negatif,10,0
Aktual Positif,0,11


### 6.1 Tabel perbandingan metrik

Bagian ini membandingkan performa dua pendekatan klasifikasi sentimen, yaitu Logistic Regression sebagai metode Machine Learning klasik dan Gemini API sebagai pendekatan berbasis Large Language Model (LLM). Evaluasi dilakukan menggunakan metrik Accuracy, Precision, Recall, dan F1-Score.

In [22]:

# Membuat tabel perbandingan hasil evaluasi
comparison_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Logistic Regression": [
        accuracy_classic,
        precision_classic,
        recall_classic,
        f1_classic
    ],
    "Gemini API": [
        accuracy_gemini,
        precision_gemini,
        recall_gemini,
        f1_gemini
    ]
})

comparison_df


,Metric,Logistic Regression,Gemini API
0,Accuracy,1.0,1.0
1,Precision,1.0,1.0
2,Recall,1.0,1.0
3,F1-Score,1.0,1.0


### 6.2 Perbandingan jumlah data evaluasi

In [ ]:

evaluation_summary = pd.DataFrame({
    "Model": ["Logistic Regression", "Gemini API"],
    "Successful Predictions": [
        len(y_pred_classic),
        len(y_pred_gemini)
    ],
    "Total Test Data": [
        len(y_test),
        len(y_test)
    ]
})

evaluation_summary


,Model,Successful Predictions,Total Test Data
0,Logistic Regression,40,40
1,Gemini API,21,40


### 6.3 Interpretasi Hasil
Berdasarkan hasil evaluasi, Logistic Regression dan Gemini API sama-sama memperoleh Accuracy, Precision, Recall, dan F1-Score sebesar 100%. Pada eksperimen ini, kedua pendekatan berhasil mengklasifikasikan seluruh data yang dievaluasi dengan benar.

Meskipun demikian, terdapat perbedaan jumlah data evaluasi. Logistic Regression berhasil memproses seluruh 40 data pengujian, sedangkan Gemini API hanya berhasil memproses 21 dari 40 data. Hal ini disebabkan oleh keterbatasan kuota permintaan API pada layanan free tier. Selain itu, ketersediaan layanan dapat dipengaruhi oleh kondisi high demand.

Dari sisi operasional, Gemini API juga membutuhkan waktu pemrosesan yang jauh lebih lama karena setiap ulasan dikirim melalui permintaan API secara terpisah. Sebaliknya, Logistic Regression dapat melakukan prediksi secara lokal tanpa bergantung pada ketersediaan layanan eksternal.

Dengan demikian, meskipun kedua metode menunjukkan metrik evaluasi yang sama pada data yang berhasil diproses, Logistic Regression memiliki keunggulan dalam hal cakupan evaluasi dan kemudahan pemrosesan pada eksperimen ini. Hasil tersebut belum cukup untuk menyimpulkan bahwa kedua model memiliki kemampuan generalisasi yang setara karena jumlah data evaluasinya berbeda.

### 6.4 Tabel perbandingan waktu pemrosesan

In [24]:

import time

# Mengukur waktu prediksi Logistic Regression
start_time = time.time()

y_pred_classic = clf.predict(X_test_vec)

classic_inference_time = time.time() - start_time

print("Jumlah data:", len(y_pred_classic))
print(f"Waktu inferensi: {classic_inference_time:.6f} detik")


Jumlah data: 40
Waktu inferensi: 0.000316 detik


In [26]:

time_comparison_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Gemini API"],
    "Successful Predictions": [40, 21],
    "Processing Time (seconds)": [
        classic_inference_time,
        538.79
    ]
})

time_comparison_df


,Model,Successful Predictions,Processing Time (seconds)
0,Logistic Regression,40,0.000316
1,Gemini API,21,538.790000


### 6.5 Narasi hasil perbandingan waktu
Berdasarkan pengukuran waktu pemrosesan, Logistic Regression membutuhkan waktu inferensi sebesar 0,000316 detik untuk memproses 40 data pengujian. Sementara itu, Gemini API membutuhkan waktu sekitar 538,79 detik untuk menghasilkan 21 prediksi yang berhasil.

Perbedaan tersebut menunjukkan bahwa Logistic Regression memiliki waktu pemrosesan yang jauh lebih singkat dalam eksperimen ini. Hal ini karena Logistic Regression melakukan prediksi secara lokal menggunakan model yang telah dilatih sebelumnya, sedangkan Gemini API memerlukan komunikasi dengan layanan eksternal untuk setiap permintaan klasifikasi.

Waktu pemrosesan Gemini juga dipengaruhi oleh jeda antarpermintaan API yang diterapkan untuk mengurangi risiko pembatasan penggunaan, serta waktu respons layanan. Oleh karena itu, angka tersebut merepresentasikan total waktu proses eksperimen, bukan murni waktu komputasi model.

Selain perbedaan waktu, eksperimen Gemini juga menghadapi keterbatasan kuota permintaan pada free tier dan potensi kendala ketersediaan layanan ketika terjadi high demand. Kondisi tersebut menyebabkan hanya 21 dari 40 data pengujian yang berhasil diproses.

## 7. Trade-off & Limitations

Pada bagian ini dilakukan analisis terhadap kelebihan dan keterbatasan
masing-masing pendekatan berdasarkan hasil eksperimen, meliputi performa
klasifikasi, waktu pemrosesan, ketergantungan layanan, serta kemudahan implementasi.

### 7.1 Perbandingan Trade-off

| Aspek | Logistic Regression | Gemini API |
|---|---|---|
| Pendekatan | Machine Learning klasik | Large Language Model (LLM) |
| Persiapan model | Memerlukan training menggunakan dataset | Tidak memerlukan training khusus pada dataset |
| Accuracy | 100% (40 data) | 100% (21 data) |
| Waktu pemrosesan | Sangat cepat, dilakukan secara lokal | Lebih lama karena menggunakan API |
| Ketergantungan internet | Tidak diperlukan saat inferensi | Memerlukan koneksi internet |
| Ketergantungan layanan eksternal | Tidak ada | Bergantung pada ketersediaan API |
| Batas penggunaan | Tidak memiliki kuota API | Memiliki batas kuota sesuai layanan |
| Biaya operasional | Relatif rendah setelah model dilatih | Bergantung pada kuota dan skema penggunaan API |
| Fleksibilitas | Bergantung pada data dan proses training | Dapat mengikuti instruksi melalui prompt |

### 7.2 Limitations

**Logistic Regression**

1. Kinerja model bergantung pada kualitas dan representasi dataset pelatihan.
2. TF-IDF hanya merepresentasikan teks berdasarkan bobot kata dan tidak secara langsung memahami konteks semantik secara mendalam.
3. Model perlu dilatih kembali apabila terdapat perubahan karakteristik data yang signifikan.

**Gemini API**

1. Penggunaan API pada free tier memiliki keterbatasan kuota permintaan.
2. Proses klasifikasi membutuhkan koneksi internet dan bergantung pada ketersediaan layanan eksternal.
3. Waktu pemrosesan lebih lama dibandingkan Logistic Regression pada eksperimen ini.
4. Proses eksperimen hanya berhasil memproses 21 dari 40 data pengujian karena keterbatasan kuota API.
5. Layanan API dapat mengalami keterlambatan atau pembatasan ketika terjadi high demand.

**Keterbatasan Eksperimen**

1. Dataset yang digunakan hanya terdiri dari 200 ulasan pelanggan.
2. Jumlah data evaluasi kedua pendekatan berbeda, yaitu 40 data untuk Logistic Regression dan 21 data untuk Gemini API.
3. Hasil evaluasi 100% pada kedua pendekatan belum cukup untuk membuktikan kemampuan generalisasi pada dataset yang lebih besar dan beragam.
4. Pengukuran waktu Gemini mencakup jeda antarrequest dan komunikasi API, sedangkan pengukuran Logistic Regression hanya mencakup waktu inferensi lokal.

### 7.3 Interpretasi Trade-off

Berdasarkan eksperimen, Logistic Regression lebih unggul dari sisi efisiensi
pemrosesan, kemudahan penggunaan secara lokal, dan kemampuan memproses seluruh
data pengujian tanpa batasan kuota API.

Di sisi lain, Gemini API menawarkan pendekatan klasifikasi berbasis model bahasa
yang tidak memerlukan proses training khusus menggunakan dataset penelitian.
Namun, penggunaannya memiliki ketergantungan terhadap koneksi internet,
ketersediaan layanan, dan batas kuota API.

Walaupun kedua pendekatan memperoleh nilai evaluasi sebesar 100%, hasil tersebut
belum dapat dibandingkan secara sepenuhnya setara karena jumlah data evaluasi
Gemini lebih sedikit. Oleh karena itu, pemilihan pendekatan perlu mempertimbangkan
tidak hanya performa klasifikasi, tetapi juga kebutuhan operasional dan sumber daya.

## 8. Technical Recommendation

Berdasarkan hasil eksperimen dan analisis trade-off, bagian ini memberikan
rekomendasi pendekatan yang paling sesuai untuk implementasi klasifikasi
sentimen ulasan pelanggan.

### 8.1 Recommended Approach

Berdasarkan hasil eksperimen, Logistic Regression dengan representasi fitur
TF-IDF direkomendasikan sebagai pendekatan utama untuk implementasi klasifikasi
sentimen ulasan pelanggan pada skenario penelitian ini.

Terdapat beberapa alasan yang mendukung rekomendasi tersebut:

1. **Efisiensi pemrosesan:** Logistic Regression mampu memproses 40 data pengujian
   dalam waktu sekitar 0,000316 detik, jauh lebih cepat dibandingkan total
   pemrosesan Gemini API yang tercatat sebesar 538,79 detik untuk 21 prediksi.

2. **Cakupan evaluasi:** Logistic Regression berhasil memproses seluruh 40 data
   pengujian, sedangkan Gemini API hanya berhasil memproses 21 data akibat
   keterbatasan kuota API.

3. **Kemandirian layanan:** Model Logistic Regression dapat dijalankan secara
   lokal tanpa bergantung pada koneksi internet maupun ketersediaan layanan API
   eksternal.

4. **Performa klasifikasi:** Logistic Regression memperoleh Accuracy,
   Precision, Recall, dan F1-Score sebesar 100% pada 40 data pengujian.

Meskipun demikian, Gemini API tetap memiliki potensi untuk digunakan sebagai
alternatif klasifikasi sentimen, terutama ketika pengembang ingin memanfaatkan
kemampuan Large Language Model tanpa harus melakukan training model sendiri.

Gemini API juga dapat digunakan untuk eksperimen berbasis prompt atau sebagai
pembanding terhadap model Machine Learning klasik. Namun, penggunaannya perlu
mempertimbangkan kuota API, waktu respons, ketergantungan internet, dan
kemungkinan keterbatasan layanan ketika terjadi high demand.

### 8.2 Final Conclusion

Eksperimen ini membandingkan dua pendekatan klasifikasi sentimen, yaitu
Logistic Regression dengan TF-IDF dan Gemini API.

Kedua pendekatan memperoleh Accuracy, Precision, Recall, dan F1-Score sebesar
100% pada data yang berhasil dievaluasi. Logistic Regression dievaluasi
menggunakan 40 data, sedangkan Gemini API hanya berhasil memproses 21 data
dari total 40 data pengujian.

Dari sisi efisiensi, Logistic Regression menunjukkan waktu inferensi yang
jauh lebih singkat dan tidak memiliki ketergantungan terhadap layanan eksternal.
Sementara itu, Gemini API menghadapi keterbatasan kuota penggunaan dan
membutuhkan waktu pemrosesan lebih lama.

Dengan mempertimbangkan hasil eksperimen, Logistic Regression menjadi
rekomendasi utama untuk skenario klasifikasi sentimen pada penelitian ini.
Namun, hasil evaluasi yang diperoleh masih perlu diuji menggunakan dataset
yang lebih besar dan beragam untuk mengetahui kemampuan generalisasi model
secara lebih menyeluruh.